# Task C — amortization at the full 74-constraint level, both arms on one device

Pre-registered: `taskc/DECISIONS.md` §5 (disjoint draws), §11 (h_ψ), §13 (unit test). This notebook **loads the frozen P_θ checkpoint** (`models/frozen_ptheta/`, committed to the repo — the same model as §5.4 and the rest of the paper; it is not retrained here), then solves the dual, trains h_ψ and runs **both arms of the matched-compute comparison on this device**, so the comparison is not mixing hardware:

- **arm W (weighted)** — P_θ draws + importance weights from β*, and multinomial resampling to produce unweighted paths;
- **arm A (amortized)** — the corrected sampler with the ε-hook `−√(1−ᾱ)·∇log h_ψ`.

Compute is booked for each arm (network evaluations and wall-clock, including h_ψ's training cost amortized over the draw) so "when is it worth converting weights into a sampler" is answered with numbers rather than an impression.

Constraints: the C3 calibration set (53 columns) **plus** the 21 held-out columns is 74 in total; the dual is solved on the 53 calibrated ones and the 21 held-out ones are scored, never calibrated. Writes to Drive after every stage. A100 runtime ≈ 25–30 min.

In [ ]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch --all
!git checkout v2_code
!git log --oneline -1

In [ ]:
import os, sys, json, math, pickle, time
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, torch
import matplotlib.pyplot as plt
from dataclasses import asdict, replace
import taskc
from taskc.config import CFG, FROZEN, frozen
from taskc.data import build_training_set, make_loader, reference_paths
from taskc.ptheta import (make_schedule, build_model, train_ptheta, save_checkpoint, load_checkpoint,
                          sample_ptheta, save_draw, load_draw, Schedule)
from taskc.gate import run_gate, print_report, summary_dict
from config import q_params, CONSTRAINT_LEVELS, EXOTICS      # taskb

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ddpm_option_pricing"
else:
    DRIVE = "artifacts_colab_local"
os.makedirs(DRIVE, exist_ok=True)

## Config

`FROZEN` (not `CFG`) carries the recipe the checkpoint was trained with: 450 epochs, cosine lr 1e-3 → 1e-5, EMA 0.999, batch 512. `CFG` keeps decay/EMA off so the earlier ladder rungs stay reproducible, so a bare `replace(CFG, …)` would silently train the constant-lr model that fails the gate.

In [ ]:
RUN_TAG = "amort74"
CKPT_IN = "models/frozen_ptheta/ptheta_mlp21.pt"     # the frozen P_theta, committed to the repo
OUT = os.path.join(DRIVE, "artifacts_" + RUN_TAG); os.makedirs(OUT, exist_ok=True)
RUN = frozen(artifact_dir=OUT)        # 450 epochs, cosine lr decay, EMA -- matches the checkpoint
LEVEL = "C3"
res = dict(tag=RUN_TAG, device=DEVICE, device_name=(torch.cuda.get_device_name(0) if DEVICE=="cuda" else DEVICE),
           config=asdict(RUN), level=LEVEL, timings={}, stages={})
def save():
    json.dump(res, open(os.path.join(OUT, "amortization.json"), "w"), indent=1, default=float)
save(); print("writing to", OUT)
print(f"draws: A {RUN.draws['A'].n:,} (dual) | B {RUN.draws['B'].n:,} (h_psi targets) | C {RUN.draws['C'].n:,} (evaluation) | solve draw {RUN.solve_draw.n:,}")

## Stage 1 — load the frozen P_θ and re-run the step-2 gate as a device-consistency check

The checkpoint passed this gate on MPS; re-running it here checks that the device does not move the statistics, and the gate is expected to pass.

The gate's thresholds are calibrated for 10⁵-path draws (`DECISIONS.md` §7, §14): at smaller draw sizes G2b and G1-tail fail even simulator-vs-simulator, so do not shrink `draws` here.

In [ ]:
ts = build_training_set(RUN); sched = make_schedule(RUN, device=DEVICE)
print(f"training set {ts.z.shape}, rejected {ts.n_rejected}, max|z| {ts.max_abs_z:.2f}; sampler t_start {sched.t_start}")
model, std_ck, ck_cfg, ck_extra = load_checkpoint(CKPT_IN, device=DEVICE)
assert std_ck == ts.std, f"checkpoint standardizer {std_ck} != training-set standardizer {ts.std}"
assert ck_cfg["epochs"] == 450 and ck_cfg["lr_decay"] and ck_cfg["ema"], f"checkpoint was not trained with the frozen recipe: {ck_extra}"
print("loaded frozen P_theta:", ck_extra)
res["stages"]["checkpoint"] = dict(path=CKPT_IN, extra=ck_extra, standardizer=std_ck.state_dict())
res["timings"]["train_s"] = 0.0                      # not retrained here; see DECISIONS.md section 9
t0 = time.time(); A = sample_ptheta(model, sched, RUN.draws["A"].n, RUN.draws["A"].seed, RUN, DEVICE, verbose=False); save_draw(RUN, "A", A)
B = sample_ptheta(model, sched, RUN.draws["B"].n, RUN.draws["B"].seed, RUN, DEVICE, verbose=False); save_draw(RUN, "B", B)
C = sample_ptheta(model, sched, RUN.draws["C"].n, RUN.draws["C"].seed, RUN, DEVICE, verbose=False); save_draw(RUN, "C", C)
res["timings"]["draws_ABC_s"] = time.time() - t0
res["stages"]["draws"] = {k: dict(n=int(v.z.shape[0]), seed=v.seed, rejected=v.n_rejected, seconds=v.seconds, device=v.device) for k, v in (("A",A),("B",B),("C",C))}
gate = run_gate(A.z, ts.std, RUN, ref=reference_paths(RUN)); print_report(gate, columns=False)
res["stages"]["gate"] = summary_dict(gate); save()
assert gate.passed, "GATE FAILED on this device -- the checkpoint passed it on MPS, so investigate the device before proceeding"

## Stage 2 — dual on the 10⁶ solve draw; L* on B and C

In [ ]:
from taskc.dual import solve_level, evaluate_on
from taskc.ptheta import draw_path
t0 = time.time()
S6 = sample_ptheta(model, sched, RUN.solve_draw.n, RUN.solve_draw.seed, RUN, DEVICE, verbose=False); save_draw(RUN, "S6", S6)
res["timings"]["solve_draw_s"] = time.time() - t0
q = q_params(); pS6 = ts.std.to_paths(S6.z, "S6")
t0 = time.time(); tilt, r, cs = solve_level(LEVEL, pS6, q); res["timings"]["dual_s"] = time.time() - t0
pC = ts.std.to_paths(C.z, "C"); eC = evaluate_on(tilt, pC, q); w = eC["w"]
print(f"dual {LEVEL}: m={cs.m} |beta_raw|={np.linalg.norm(tilt.beta_raw):.2f} screen min margin {r.screen['margin'].min():.3f} ESS_S6 {r.ess_frac*100:.2f}% ESS_C {eC['ess_frac']*100:.2f}% E_C[L*] {eC['E_L']:.4f} ({res['timings']['dual_s']:.0f}s)")
pickle.dump(tilt, open(os.path.join(OUT, f"tilt_{LEVEL}.pkl"), "wb"))
res["stages"]["dual"] = dict(m=cs.m, beta_raw_norm=float(np.linalg.norm(tilt.beta_raw)), screen_margin=float(r.screen["margin"].min()),
                             ess_solve=float(r.ess_frac), ess_C=float(eC["ess_frac"]), E_C_L=float(eC["E_L"]), converged=bool(r.converged)); save()

## Stage 3 — h_ψ on draw B

In [ ]:
from taskc.hnet import HNet, train_hnet, tower_curve, h0_vs_L, grad_ratio_curve, save_hnet
from taskc.run_hpsi import targets_for
LB = targets_for(tilt, ts.std, B.z, q); LC = targets_for(tilt, ts.std, C.z, q)
hnet = HNet(RUN.data_dim, 256, 32, 0.05)
t0 = time.time(); hlog = train_hnet(hnet, B.z, LB, sched, device=DEVICE, epochs=100, verbose=False); res["timings"]["hpsi_s"] = time.time() - t0
save_hnet(os.path.join(OUT, f"hpsi_{LEVEL}.pt"), hnet, dict(level=LEVEL, device=DEVICE, E_L_B=float(LB.mean())))
tlist = [0,5,10,20,40,60,80,100,150,200,250,300,400,500,600,700,800,900,950,980,998]
tower = tower_curve(hnet, C.z, sched, tlist, device=DEVICE); h0 = h0_vs_L(hnet, C.z, LC, sched, device=DEVICE)
print(f"h_psi: {res['timings']['hpsi_s']/60:.1f} min | targets on B E={LB.mean():.4f} [{LB.min():.3f},{LB.max():.3f}] | tower max |E[h]-1| = {max(abs(tower[t]['mean']-1) for t in tlist):.4f} | h0 vs L*: corr {h0['corr']:.4f} R2 {h0['r2']:.4f} | floor max {max(hlog.floor_frac):.1e}")
res["stages"]["hpsi"] = dict(E_L_B=float(LB.mean()), L_min=float(LB.min()), L_max=float(LB.max()), tower={str(t): tower[t] for t in tlist},
                             h0=h0, mse=hlog.epoch_loss, floor_max=max(hlog.floor_frac)); save()

## Stage 4 — the two arms, matched compute

Arm W: draw C (already sampled) + weights, then multinomial resampling to n paths.
Arm A: the corrected sampler, n paths. Both scored on the same 74 columns (53 calibrated + 21 held out) and the three exotics.

In [ ]:
from taskc.smt import sample_smt, sliced_wasserstein, resample_weighted, compute_budget
from taskc.dual import baseline_on
from constraints import build, build_heldout
from config import HELDOUT_TESTFUNS, HELDOUT_VANILLAS
from projection import wmean, wmean_se
import evaluation as ev
N_EVAL = RUN.draws["C"].n
spec = CONSTRAINT_LEVELS[LEVEL]
# arm A
t0 = time.time(); SMT = sample_smt(model, hnet, sched, N_EVAL, 2001, RUN, DEVICE, verbose=False); res["timings"]["smt_draw_s"] = time.time() - t0
save_draw(replace(RUN, artifact_dir=OUT), f"smt_{LEVEL}", SMT)
pS = ts.std.to_paths(SMT.z, "SMT")
# arm W (resampled), same n
t0 = time.time(); zW = resample_weighted(C.z, w, N_EVAL, seed=2002); res["timings"]["resample_s"] = time.time() - t0
pW = ts.std.to_paths(zW, "weighted-resampled")
budget = dict(
  armW=compute_budget(RUN.draws["C"].n + RUN.solve_draw.n, sched.t_start + 1, res["timings"]["draws_ABC_s"] * (1/3) + res["timings"]["solve_draw_s"] + res["timings"]["dual_s"] + res["timings"]["resample_s"]),
  armA=compute_budget(N_EVAL, sched.t_start + 1, res["timings"]["smt_draw_s"] + res["timings"]["hpsi_s"] + res["timings"]["draws_ABC_s"] * (1/3)))
res["stages"]["budget"] = budget
print("compute: arm W", budget["armW"], "\n         arm A", budget["armA"])
csS, csC = build(pS, spec["testfuns"], spec["vanillas"], q), build(pC, spec["testfuns"], spec["vanillas"], q)
hoS, hoC = build_heldout(pS, HELDOUT_TESTFUNS, HELDOUT_VANILLAS, q), build_heldout(pC, HELDOUT_TESTFUNS, HELDOUT_VANILLAS, q)
mS, seS = csS.G.mean(0), csS.G.std(0, ddof=1)/np.sqrt(csS.n)
mW = np.array([wmean(csC.G[:,j], w) for j in range(csC.m)]); seW = np.array([wmean_se(csC.G[:,j], w) for j in range(csC.m)])
devS, devW, dSW = (mS-csS.c)/seS, (mW-csC.c)/seW, (mS-mW)/np.hypot(seS,seW)
mhS = hoS.G.mean(0); mhW = np.array([wmean(hoC.G[:,j], w) for j in range(hoC.m)])
dh = (mhS-mhW)/np.hypot(hoS.G.std(0,ddof=1)/np.sqrt(hoS.n), np.array([wmean_se(hoC.G[:,j], w) for j in range(hoC.m)]))
print(f"\ncalibrated (53): SMT vs c max {np.abs(devS).max():.2f} SE, within 2 SE {int((np.abs(devS)<=2).sum())}/{csS.m} | weighted vs c max {np.abs(devW).max():.2f} SE | SMT vs weighted max {np.abs(dSW).max():.2f} SE, within 2 SE {int((np.abs(dSW)<=2).sum())}/{csS.m}")
print(f"held-out  (21): SMT vs weighted max {np.abs(dh).max():.2f} SE, within 2 SE {int((np.abs(dh)<=2).sum())}/{hoS.m}")
exS, exW, exR = ev.price_exotics(pS), eC["exotics"], ev.price_exotics(pW)
for k in EXOTICS:
    (a,sa),(b,sb),(c2,sc) = exS[k], exW[k], exR[k]
    print(f"  {k:28s} SMT {a:.4f}+-{sa:.4f}  weighted {b:.4f}+-{sb:.4f} ({(a-b)/np.hypot(sa,sb):+.2f} SE)  resampled {c2:.4f}+-{sc:.4f}")
sw = dict(null=sliced_wasserstein(A.z, C.z), signal=sliced_wasserstein(C.z, C.z, wX=w), test=sliced_wasserstein(C.z, SMT.z, wX=w), resampled_vs_smt=sliced_wasserstein(zW, SMT.z))
print("sliced Wasserstein:", {k: round(v,5) for k,v in sw.items()})
res["stages"]["arms"] = dict(calibrated=dict(smt_vs_c_max=float(np.abs(devS).max()), smt_vs_c_within2=int((np.abs(devS)<=2).sum()),
                                             w_vs_c_max=float(np.abs(devW).max()), smt_vs_w_max=float(np.abs(dSW).max()), smt_vs_w_within2=int((np.abs(dSW)<=2).sum()), m=int(csS.m)),
                             heldout=dict(max=float(np.abs(dh).max()), within2=int((np.abs(dh)<=2).sum()), m=int(hoS.m)),
                             exotics={k: dict(smt=[float(exS[k][0]),float(exS[k][1])], weighted=[float(exW[k][0]),float(exW[k][1])], resampled=[float(exR[k][0]),float(exR[k][1])]) for k in EXOTICS},
                             sw=sw, ess_C=float(eC["ess_frac"]), rejections_smt=int(SMT.n_rejected)); save()

## Stage 5 — variance at matched compute, and the step-count series

In [ ]:
from taskc.smt import sample_strided
reps = 5
pay = {k: [] for k in EXOTICS}
for i in range(reps):
    zi = resample_weighted(C.z, w, N_EVAL, seed=3000+i); pi = ts.std.to_paths(zi)
    for k in EXOTICS: pay[k].append(ev.price_exotics(pi)[k][0])
print("arm W resampling spread over 5 reps (same weighted pool):", {k: round(float(np.std(v, ddof=1)), 5) for k, v in pay.items()})
res["stages"]["resample_spread"] = {k: dict(mean=float(np.mean(v)), sd=float(np.std(v, ddof=1))) for k, v in pay.items()}
sd = csC.G.std(0); series = {}
for K in (sched.t_start+1, 500, 250, 100, 50):
    dS = sample_strided(model, sched, 20_000, K, seed=4000+K, cfg=RUN, device=DEVICE, hnet=hnet)
    gS = build(ts.std.to_paths(dS.z), spec["testfuns"], spec["vanillas"], q).G
    series[K] = float(np.abs((gS.mean(0)-csS.c)/sd).max()); print(f"  K={K:4d}: SMT max |E[g]-c|/sd = {series[K]:.4f}")
res["stages"]["step_series"] = {str(k): v for k, v in series.items()}
res["timings"]["total_s"] = sum(v for v in res["timings"].values()); save()
print(f"\ntotal {res['timings']['total_s']/60:.1f} min -> {OUT}")

Results in `amortization.json`; numbers go to `DECISIONS.md` §13 before the manuscript is touched.